# Lab01 - sprint 2
## random 01: оценка качества при заданной крепости
### Description

Прогноз kNN даёт среднюю оценку. Генератор выдаёт отдельные целые оценки с частотами,
наблюдаемыми у ближайших по крепости вин обучающей выборки. k = 720, совпадения на границе сохраняем.
Это модель условного распределения, а не воспроизведение дегустации конкретного вина.
k выбран для среднего прогноза; оптимальность для всего распределения не установлена.



### math desc - метод обратной функции распределения

Каждой оценке выделяем интервал от 0 до 1. Его длина равна доле этой оценки среди соседей.
Равномерное случайное число выбирает интервал, а вместе с ним - оценку.

**Пример.** Числа в таблице условные и поясняют алгоритм.

| Оценка | Вероятность | Интервал для U |
|---|---:|---|
| 5 | 0,20 | [0; 0,20) |
| 6 | 0,50 | [0,20; 0,70) |
| 7 | 0,30 | [0,70; 1) |

Если U = 0,54, генератор возвращает 6. Вероятность получить шестёрку равна длине её интервала:

$$
P(Q^*=6)=P(0{,}20\le U<0{,}70)=0{,}70-0{,}20=0{,}50.
$$

**Распределение по соседям.** При заданной крепости x выбираем m соседей из train.
Различные оценки обозначим q₁ < … < qᵣ; оценка qⱼ встречается nⱼ раз.
Тогда её оценённая вероятность:

$$
p_j(x)=\frac{n_j}{m},\qquad \sum_{j=1}^{r}p_j(x)=1.
$$

Берём k = 720, но m может быть больше: все совпадения расстояний на границе включаем целиком.

**Правило генерации.** Накапливаем вероятности:

$$
C_0=0,\qquad C_j=\sum_{i=1}^{j}p_i(x),\qquad C_r=1.
$$

Для равномерного U на [0; 1) возвращаем оценку qⱼ, если:

$$
C_{j-1}\le U<C_j.
$$

**Обоснование.** У равномерного распределения вероятность попасть в интервал равна его длине:

$$
P(Q^*=q_j\mid x)=P(C_{j-1}\le U<C_j)=C_j-C_{j-1}.
$$

Вычитая две накопленные суммы, получаем:

$$
C_j-C_{j-1}
=\left(\sum_{i=1}^{j-1}p_i(x)+p_j(x)\right)-\sum_{i=1}^{j-1}p_i(x)
=p_j(x).
$$

Следовательно:

$$
\boxed{P(Q^*=q_j\mid x)=p_j(x).}
$$

Генератор воспроизводит заданное распределение оценок. В коде ищем первую накопленную
вероятность, строго большую U: `searchsorted(cdf, U, side='right')`.
Это учитывает используемый интервал [0; 1) и пропускает оценки с нулевой вероятностью.
Базовые псевдослучайные равномерные числа получаем из NumPy.

**Связь с прогнозом kNN.** Среднее генератора равно среднему оценок соседей:

$$
\mu(x)=E[Q^*\mid x]=\sum_{j=1}^{r}q_jp_j(x)
=\frac{1}{m}\sum_{i\in N(x)}q_i.
$$

Дисперсия показывает разброс сгенерированных оценок:

$$
\operatorname{Var}(Q^*\mid x)=\sum_{j=1}^{r}(q_j-\mu(x))^2p_j(x).
$$

Эти формулы задают значения для проверки генератора. В конечной случайной выборке
среднее, дисперсия и частоты могут немного отличаться от них.
Доказательство относится к распределению соседей; соответствие новым реальным винам проверяется отдельно.

**Источник:** L. Devroye, *Non-Uniform Random Variate Generation* (1986), глава II, §2, с. 27-31.
[Текст книги](https://luc.devroye.org/LucDevroye-NonUniformRandomVariateGeneration-10.1007_978-1-4613-8643-8-1986.pdf).


### code

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'pyproject.toml').is_file())
sys.path.insert(0,str(ROOT/'src'))
from wine_stats.data import load_wines
from wine_stats.split import split_wines
from wine_stats.generator import quality_distribution, generate_quality
df=load_wines(ROOT/'data/raw')
manifest=pd.read_csv(ROOT/'data/processed/alcohol_quality_split.csv')
pd.testing.assert_frame_equal(manifest,split_wines(df,seed=42))
train=df.loc[manifest['split'].eq('train')]
ALCOHOL=12.
scores,p=quality_distribution(train.alcohol,train.quality,ALCOHOL,k=720)
print(generate_quality(train.alcohol,train.quality,ALCOHOL,20,k=720,seed=42))
display(pd.DataFrame({'quality':scores,'probability':p}))


[7 6 7 7 6 8 7 7 6 6 6 8 7 7 6 6 6 5 7 7]


,quality,probability
0,3,0.002519
1,4,0.013854
2,5,0.075567
3,6,0.505038
4,7,0.322418
5,8,0.078086
6,9,0.002519


### tests

Сравниваем генерацию с заданным распределением соседей. Это проверка реализации,
а не независимая проверка модели на новых винах.


In [2]:
sample=generate_quality(train.alcohol,train.quality,ALCOHOL,100000,k=720,seed=42)
mu=scores@p
comparison=pd.DataFrame({
    'metric':['mean','variance'],
    'neighbors_distribution':[mu,((scores-mu)**2)@p],
    'generated':[sample.mean(),sample.var()]})
display(comparison)
comparison.to_csv(ROOT/'reports/tables/generator_smoke_metrics.csv',index=False)


,metric,neighbors_distribution,generated
0,mean,6.375315,6.377880
1,variance,0.670222,0.672827


### resume

При 12% спирта среднее распределения соседей - 6,375 балла, генерации - 6,378.
Дисперсии - 0,670 и 0,673 балла². Небольшое расхождение связано со случайной генерацией.
Алгоритм прошёл тесты границ интервалов, частот, воспроизводимости и допустимых значений.
Сравнение с реальными данными приведено в `04_generator_comparison.ipynb`.

## Lab resume
### cat_mark_table

| cat_id | notebook_link | coeff | comment |
|---|---|---|---|
| - | [Генератор](01_generator.ipynb) | - | Первая реализация и тесты; добавлено обоснование метода; проверка на test приведена в следующем разделе. |
